In [1]:
import pandas as pd
from sqlalchemy import create_engine

df = pd.read_csv('EmendasParlamentares.csv', sep = ';', encoding='latin-1')
print(f"O arquivo possui {len(df)} linhas e {len(df.columns)} colunas.")
df.head()

O arquivo possui 89737 linhas e 28 colunas.


C:\Users\hboni\AppData\Local\Temp\ipykernel_16444\1615024034.py:4: DtypeWarning: Columns (0: Código da Emenda, 1: Código do Autor da Emenda, 2: Número da emenda) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('EmendasParlamentares.csv', sep = ';', encoding='latin-1')


,Código da Emenda,Ano da Emenda,Tipo de Emenda,Código do Autor da Emenda,Nome do Autor da Emenda,Número da emenda,Localidade de aplicação do recurso,Código Município IBGE,Município,Código UF IBGE,...,Código Ação,Nome Ação,Código Plano Orçamentário,Nome Plano Orçamentário,Valor Empenhado,Valor Liquidado,Valor Pago,Valor Restos A Pagar Inscritos,Valor Restos A Pagar Cancelados,Valor Restos A Pagar Pagos
0,Sem informação,2014,Emenda Individual - Transferências com Finalid...,S/I,Sem informação,S/I,MUNDO NOVO - MS,5005681,MUNDO NOVO,5000000,...,5450,IMPLANTACAO E MODERNIZACAO DE INFRAESTRUTURA P...,0000,IMPLANTACAO E MODERNIZACAO DE INFRAESTRUTURA P...,"645000,00","0,00","0,00","0,00","0,00","645000,00"
1,Sem informação,2014,Emenda Individual - Transferências com Finalid...,S/I,Sem informação,S/I,SÃO GONÇALO - RJ,3304904,SÃO GONÇALO,3300000,...,20RP,APOIO A INFRAESTRUTURA PARA A EDUCACAO BASICA,0000,INFRAESTRUTURA PARA A EDUCACAO BASICA - DESPES...,"179919,36","0,00","0,00","0,00","179919,36","0,00"
2,Sem informação,2014,Emenda Individual - Transferências com Finalid...,S/I,Sem informação,S/I,CASTRO - PR,4104907,CASTRO,4100000,...,8581,ESTRUTURACAO DA REDE DE SERVICOS DE ATENCAO PR...,0000,ESTRUTURACAO DA REDE DE SERVICOS DE ATENCAO BA...,"858000,00","0,00","0,00","0,00","0,00","858000,00"
3,Sem informação,2014,Emenda Individual - Transferências com Finalid...,S/I,Sem informação,S/I,GOIÁS (UF),Sem informação,Sem informação,5200000,...,8535,ESTRUTURACAO DE UNIDADES DE ATENCAO ESPECIALIZ...,0000,ESTRUTURACAO DE UNIDADES DE ATENCAO ESPECIALIZ...,"8281302,28","0,00","0,00","0,00","400000,00","7881302,28"
4,Sem informação,2014,Emenda Individual - Transferências com Finalid...,S/I,Sem informação,S/I,BOM JESUS - RN,2401701,BOM JESUS,2400000,...,5450,IMPLANTACAO E MODERNIZACAO DE INFRAESTRUTURA P...,0000,IMPLANTACAO E MODERNIZACAO DE INFRAESTRUTURA P...,"292500,00","0,00","0,00","0,00","0,00","292500,00"


In [2]:
df ['Valor Empenhado'] = df ['Valor Empenhado'].astype(str).str.replace(',','', regex=False).str.replace('.','.', regex=False)
df ['Valor Empenhado'] = pd.to_numeric(df ['Valor Empenhado'], errors='coerce').fillna(0)

df ['Nome do Autor da Emenda'] = df ['Nome do Autor da Emenda'].str.upper().str.strip()
df ['Nome Função'] = df ['Nome Função'].str.upper().str.strip()

import numpy as np
df = df.replace('Sem Informação', np.nan)

colunas_selecionadas = ['Ano da Emenda', 'Tipo de Emenda', 'Nome do Autor da Emenda', 'Município', 'UF', 'Nome Função', 'Valor Empenhado', ]
df_final = df[colunas_selecionadas].copy()

print("Limpeza Concluida com Sucesso!")
df_final.head()

Limpeza Concluida com Sucesso!


,Ano da Emenda,Tipo de Emenda,Nome do Autor da Emenda,Município,UF,Nome Função,Valor Empenhado
0,2014,Emenda Individual - Transferências com Finalid...,SEM INFORMAÇÃO,MUNDO NOVO,MATO GROSSO DO SUL,DESPORTO E LAZER,64500000
1,2014,Emenda Individual - Transferências com Finalid...,SEM INFORMAÇÃO,SÃO GONÇALO,RIO DE JANEIRO,EDUCAÇÃO,17991936
2,2014,Emenda Individual - Transferências com Finalid...,SEM INFORMAÇÃO,CASTRO,PARANÁ,SAÚDE,85800000
3,2014,Emenda Individual - Transferências com Finalid...,SEM INFORMAÇÃO,Sem informação,GOIÁS,SAÚDE,828130228
4,2014,Emenda Individual - Transferências com Finalid...,SEM INFORMAÇÃO,BOM JESUS,RIO GRANDE DO NORTE,DESPORTO E LAZER,29250000


In [3]:
engine = create_engine('sqlite:///transparencia_emendas.db')
df_final.to_sql('emendas_parlamentares', con=engine, if_exists='replace', index=False)

print("Dados inseridos no banco de dados com sucesso!")

Dados inseridos no banco de dados com sucesso!


In [4]:
termos_para_remover = ['BANCADA', 'COM.', 'RELATOR', 'SEM INFORMAÇÃO']
filtro = ~df_final['Nome do Autor da Emenda'].str.contains('|'.join(termos_para_remover), case=False, na=False)
df_parlamentares = df_final[filtro]

top_autores = df_parlamentares.groupby('Nome do Autor da Emenda')['Valor Empenhado'].sum().sort_values(ascending=False)
print("Analise Concluida com Sucesso!\n")
for nome, valor in top_autores.head(10).items():
    print(f"{nome}: R$ {valor:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))

Analise Concluida com Sucesso!

RODRIGO PACHECO: R$ 33.662.911.462,00
PAULO PAIM: R$ 31.904.593.596,00
MARA GABRILLI: R$ 31.609.721.763,00
JADER BARBALHO: R$ 31.067.108.869,00
DAVI ALCOLUMBRE: R$ 30.984.356.386,00
LUIS CARLOS HEINZE: R$ 30.966.661.873,00
MARCOS ROGERIO: R$ 30.621.065.862,00
SERGIO PETECAO: R$ 30.578.188.781,00
ESPERIDIAO AMIN: R$ 30.452.530.004,00
ELIZIANE GAMA: R$ 30.302.837.981,00


In [5]:
termos_para_remover_uf = ['SEM INFORMAÇÃO', 'Múltiplo']
filtro_uf = ~df_final['UF'].str.contains('|'.join(termos_para_remover_uf), case=False, na=False)
df_uf = df_final[filtro_uf]

verba_por_uf = df_uf.groupby('UF')['Valor Empenhado'].sum().sort_values(ascending=False)
print("Analise Concluida com Sucesso!\n")
for nome, valor in verba_por_uf.head(10).items():
    print(f"{nome}: R$ {valor:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))

Analise Concluida com Sucesso!

SÃO PAULO: R$ 2.000.928.130.534,00
MINAS GERAIS: R$ 1.601.976.810.142,00
BAHIA: R$ 1.280.292.214.169,00
RIO DE JANEIRO: R$ 1.269.865.757.774,00
RIO GRANDE DO SUL: R$ 1.060.134.348.946,00
PARANÁ: R$ 978.005.335.736,00
CEARÁ: R$ 873.595.680.192,00
PERNAMBUCO: R$ 868.965.973.570,00
MARANHÃO: R$ 771.066.087.282,00
GOIÁS: R$ 744.549.168.756,00
